In [ ]:
from pathlib import Path
from typing import List

import pandas as pd
from Pipeline.Global.Plotting import Plotting

In [ ]:
def process_and_calculate_rigorous(folder_path: str) -> pd.DataFrame:
    final_info: List[pd.DataFrame] = []
    trace_dir = Path(folder_path)

    for file_path in trace_dir.glob("*.csv"):
        df_trace = pd.read_csv(file_path)
        if df_trace.empty:
            continue

        target_iteration = df_trace['Max_Iteration'].iloc[0]
        df_converged = df_trace[df_trace['Iteration'] == target_iteration].copy()

        df_seed_agg = df_converged.groupby('Seed').agg(
            Train_Fit_Mean_by_Fold=('Train_Fitness', 'mean'),
            Train_Fit_Std_by_Fold=('Train_Fitness', 'std'),
            Val_Fit_Mean_by_Fold=('Val_Fitness', 'mean'),
            Val_Fit_Std_by_Fold=('Val_Fitness', 'std')
        ).fillna(0)

        # Intent: Compute Lower Confidence Bound (LCB) to penalize fold-level variance.
        df_seed_agg['Train_LCB'] = df_seed_agg['Train_Fit_Mean_by_Fold'] - df_seed_agg['Train_Fit_Std_by_Fold']
        df_seed_agg['Val_LCB'] = df_seed_agg['Val_Fit_Mean_by_Fold'] - df_seed_agg['Val_Fit_Std_by_Fold']

        record = {
            'expr_name': file_path.stem,
            'model_Train_Fit_Mean': df_seed_agg['Train_LCB'].mean(),
            'model_Train_Fit_Std': df_seed_agg['Train_LCB'].std(),
            'model_Val_Fit_Mean': df_seed_agg['Val_LCB'].mean(),
            'model_Val_Fit_Std': df_seed_agg['Val_LCB'].std()
        }

        record['final_train_trace'] = record['model_Train_Fit_Mean'] - record['model_Train_Fit_Std']
        record['final_val_trace'] = record['model_Val_Fit_Mean'] - record['model_Val_Fit_Std']

        final_info.append(pd.DataFrame([record]))

    return pd.concat(final_info, ignore_index=True) if final_info else pd.DataFrame()

In [ ]:
df = process_and_calculate_rigorous('../Trace History')
df

In [ ]:
def process_and_plot_traces(folder_path: str):
    trace_dir = Path(folder_path)

    csv_files = list(trace_dir.glob("*.csv"))

    for file_path in csv_files:
        experiment_name = file_path.stem

        df_trace = pd.read_csv(file_path)
        if df_trace.empty:
            raise ValueError("CSV file is empty.")
        Plotting.plot_rigorous_convergence(
            df_trace_history=df_trace,
            experiment_name=experiment_name,
            is_final_record=True
        )

In [ ]:
process_and_plot_traces("../Trace History")